In [ ]:
# Import libraries

from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

In [ ]:
# Define project paths

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_CLAIMS_DIR = PROJECT_ROOT / "data" / "01_raw" / "cms_claims"
PREPROCESSED_DIR = PROJECT_ROOT / "data" / "02_preprocessed"
FEATURES_DIR = PROJECT_ROOT / "data" / "03_features"

PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_CLAIMS_DIR:", RAW_CLAIMS_DIR)
print("PREPROCESSED_DIR:", PREPROCESSED_DIR)
print("FEATURES_DIR:", FEATURES_DIR)

In [ ]:
# Find all raw claims files

claim_files = sorted(RAW_CLAIMS_DIR.rglob("*"))

for i, file_path in enumerate(claim_files):
    if file_path.is_file():
        print(i, file_path.relative_to(PROJECT_ROOT))

In [ ]:
# Find CSV files

csv_files = sorted(RAW_CLAIMS_DIR.rglob("*.csv"))

print("CSV files found:", len(csv_files))

for i, file_path in enumerate(csv_files):
    print(i, file_path.relative_to(PROJECT_ROOT))

In [ ]:
# Auto-detect likely beneficiary, inpatient, and outpatient files

def find_files_by_keywords(files, keywords):
    matches = []

    for file_path in files:
        text = str(file_path).lower()

        if any(keyword.lower() in text for keyword in keywords):
            matches.append(file_path)

    return matches


beneficiary_matches = find_files_by_keywords(
    csv_files,
    ["beneficiary", "beneficiaries", "enrollment", "summary"],
)

inpatient_matches = find_files_by_keywords(
    csv_files,
    ["inpatient", "inp"],
)

outpatient_matches = find_files_by_keywords(
    csv_files,
    ["outpatient", "outp"],
)

print("Beneficiary matches:")
for i, file_path in enumerate(beneficiary_matches):
    print(i, file_path.relative_to(PROJECT_ROOT))

print("\nInpatient matches:")
for i, file_path in enumerate(inpatient_matches):
    print(i, file_path.relative_to(PROJECT_ROOT))

print("\nOutpatient matches:")
for i, file_path in enumerate(outpatient_matches):
    print(i, file_path.relative_to(PROJECT_ROOT))

In [ ]:
# Select files
# If this cell fails, use the printed list above and manually choose the right index.

beneficiary_file = beneficiary_matches[0]
inpatient_file = inpatient_matches[0]
outpatient_file = outpatient_matches[0]

print("Beneficiary file:", beneficiary_file.relative_to(PROJECT_ROOT))
print("Inpatient file:", inpatient_file.relative_to(PROJECT_ROOT))
print("Outpatient file:", outpatient_file.relative_to(PROJECT_ROOT))

In [ ]:
# Load raw claims datasets

beneficiary_df = pd.read_csv(beneficiary_file, low_memory=False)
inpatient_df = pd.read_csv(inpatient_file, low_memory=False)
outpatient_df = pd.read_csv(outpatient_file, low_memory=False)

print("Beneficiary shape:", beneficiary_df.shape)
print("Inpatient shape:", inpatient_df.shape)
print("Outpatient shape:", outpatient_df.shape)

display(beneficiary_df.head())
display(inpatient_df.head())
display(outpatient_df.head())

In [ ]:
# Inspect columns for each dataset

def inspect_columns(df, dataset_name):
    columns_df = pd.DataFrame({
        "dataset": dataset_name,
        "column": df.columns,
        "dtype": [df[col].dtype for col in df.columns],
        "missing_count": [df[col].isna().sum() for col in df.columns],
        "missing_pct": [(df[col].isna().mean() * 100).round(2) for col in df.columns],
    })

    return columns_df.sort_values("missing_pct", ascending=False)


beneficiary_columns = inspect_columns(beneficiary_df, "beneficiary")
inpatient_columns = inspect_columns(inpatient_df, "inpatient")
outpatient_columns = inspect_columns(outpatient_df, "outpatient")

display(beneficiary_columns)
display(inpatient_columns)
display(outpatient_columns)

In [ ]:
# Search columns by keyword

def search_columns(df, keywords):
    matches = []

    for col in df.columns:
        col_lower = col.lower()

        if any(keyword.lower() in col_lower for keyword in keywords):
            matches.append(col)

    return matches


keywords = {
    "member_id": ["desynpuf_id", "bene", "beneficiary"],
    "claim_id": ["claim", "clm"],
    "date": ["date", "dt"],
    "payment": ["payment", "pmt", "paid", "reimb"],
    "diagnosis": ["diag", "icd"],
    "provider": ["provider", "prvdr"],
}

for label, terms in keywords.items():
    print(f"\nBENEFICIARY - {label}")
    print(search_columns(beneficiary_df, terms))

    print(f"INPATIENT - {label}")
    print(search_columns(inpatient_df, terms))

    print(f"OUTPATIENT - {label}")
    print(search_columns(outpatient_df, terms))

In [ ]:
# Helper function to safely select columns

def get_first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col

    return None

In [ ]:
# Map inpatient columns to clean names

inpatient_column_map = {
    "member_id": get_first_existing_column(
        inpatient_df,
        ["DESYNPUF_ID", "BENE_ID"],
    ),
    "claim_id": get_first_existing_column(
        inpatient_df,
        ["CLM_ID"],
    ),
    "claim_start_date": get_first_existing_column(
        inpatient_df,
        ["CLM_FROM_DT", "CLM_THRU_DT"],
    ),
    "claim_end_date": get_first_existing_column(
        inpatient_df,
        ["CLM_THRU_DT"],
    ),
    "claim_payment_amount": get_first_existing_column(
        inpatient_df,
        ["CLM_PMT_AMT", "CLM_TOT_CHRG_AMT", "NCH_PRMRY_PYR_CLM_PD_AMT"],
    ),
    "provider_id": get_first_existing_column(
        inpatient_df,
        ["PRVDR_NUM"],
    ),
    "primary_diagnosis_code": get_first_existing_column(
        inpatient_df,
        ["ADMTNG_ICD9_DGNS_CD", "ICD9_DGNS_CD_1"],
    ),
}

inpatient_column_map

In [ ]:
# Map outpatient columns to clean names

outpatient_column_map = {
    "member_id": get_first_existing_column(
        outpatient_df,
        ["DESYNPUF_ID", "BENE_ID"],
    ),
    "claim_id": get_first_existing_column(
        outpatient_df,
        ["CLM_ID"],
    ),
    "claim_start_date": get_first_existing_column(
        outpatient_df,
        ["CLM_FROM_DT", "CLM_THRU_DT"],
    ),
    "claim_end_date": get_first_existing_column(
        outpatient_df,
        ["CLM_THRU_DT"],
    ),
    "claim_payment_amount": get_first_existing_column(
        outpatient_df,
        ["CLM_PMT_AMT", "CLM_TOT_CHRG_AMT", "NCH_PRMRY_PYR_CLM_PD_AMT"],
    ),
    "provider_id": get_first_existing_column(
        outpatient_df,
        ["PRVDR_NUM"],
    ),
    "primary_diagnosis_code": get_first_existing_column(
        outpatient_df,
        ["ICD9_DGNS_CD_1"],
    ),
}

outpatient_column_map